# 05 — Process Discovery
**CareFlow Process Mining | Google Colab**

Discovers a process model from the event log using PM4Py's Inductive Miner and directly-follows graph. The event log needs a case ID, activity, and usable timestamp. Check mappings and model readability before interpreting the diagrams.

In [ ]:
!pip -q install openpyxl pandas numpy matplotlib seaborn pm4py

import io, os, re, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
print("CareFlow Process Mining environment ready.")

In [ ]:
# Upload: Sepsis_Cases_Event_Log_Cleaned_for_Visualization.xlsx
uploaded = files.upload()
if not uploaded:
    raise ValueError("Please upload the processed sepsis workbook.")
workbook_name = next(iter(uploaded))
workbook_bytes = uploaded[workbook_name]
xls = pd.ExcelFile(io.BytesIO(workbook_bytes))
print("Workbook:", workbook_name)
print("Sheets:", xls.sheet_names)

required_sheet = "Event Log Visual"
if required_sheet not in xls.sheet_names:
    raise ValueError(f"Expected sheet '{required_sheet}'. Available sheets: {xls.sheet_names}")

events = pd.read_excel(io.BytesIO(workbook_bytes), sheet_name="Event Log Visual")
events.columns = [str(c).strip() for c in events.columns]
print("Event Log Visual shape:", events.shape)
display(events.head())

In [ ]:
# This workbook's event-log sheet uses these verified column names:
CASE_COL = "case_id"
ACTIVITY_COL = "activity"
TIME_COL = "timestamp"
EVENT_INDEX_COL = "event_index"
ORG_COL = "org_group"
LIFECYCLE_COL = "lifecycle_transition"

# Parse timestamp safely; invalid values become NaT and are reported.
if TIME_COL in events.columns:
    events[TIME_COL] = pd.to_datetime(events[TIME_COL], errors="coerce")

print("Case column:", CASE_COL, "| present:", CASE_COL in events.columns)
print("Activity column:", ACTIVITY_COL, "| present:", ACTIVITY_COL in events.columns)
print("Timestamp column:", TIME_COL, "| present:", TIME_COL in events.columns)

## 1. Prepare PM4Py event log

In [ ]:
if not all(col in events.columns for col in [CASE_COL, ACTIVITY_COL, TIME_COL]):
    raise ValueError("Required case/activity/timestamp columns are missing. Check the workbook schema.")

pm_df = events[[CASE_COL, ACTIVITY_COL, TIME_COL]].copy()
pm_df.columns = ["case:concept:name", "concept:name", "time:timestamp"]
pm_df["time:timestamp"] = pd.to_datetime(pm_df["time:timestamp"], errors="coerce")
pm_df = pm_df.dropna(subset=["case:concept:name", "concept:name", "time:timestamp"]).copy()
pm_df["case:concept:name"] = pm_df["case:concept:name"].astype(str)
pm_df["concept:name"] = pm_df["concept:name"].astype(str)
pm_df = pm_df.sort_values(["case:concept:name", "time:timestamp"])
print("Usable events:", len(pm_df))
print("Cases:", pm_df["case:concept:name"].nunique())
print("Activities:", pm_df["concept:name"].nunique())
display(pm_df.head())

## 2. Create PM4Py event log and discover Petri net

In [ ]:
import pm4py
from pm4py.objects.conversion.log import converter as log_converter
from pm4py.algo.discovery.inductive import algorithm as inductive_miner
from pm4py.visualization.petri_net import visualizer as pn_visualizer
from IPython.display import Image, display as ipy_display

parameters = {
    log_converter.Variants.TO_EVENT_LOG.value.Parameters.CASE_ID_KEY: "case:concept:name"
}
event_log = log_converter.apply(
    pm_df,
    variant=log_converter.Variants.TO_EVENT_LOG,
    parameters=parameters
)
net, initial_marking, final_marking = inductive_miner.apply(event_log)
gviz = pn_visualizer.apply(net, initial_marking, final_marking)
pn_visualizer.save(gviz, "discovered_process_model.png")
ipy_display(Image(filename="discovered_process_model.png"))

## 3. Discover a directly-follows graph

In [ ]:
from pm4py.algo.discovery.dfg import algorithm as dfg_discovery
from pm4py.visualization.dfg import visualizer as dfg_visualizer

dfg = dfg_discovery.apply(event_log)
dfg_gviz = dfg_visualizer.apply(dfg, log=event_log)
dfg_visualizer.save(dfg_gviz, "directly_follows_graph.png")
ipy_display(Image(filename="directly_follows_graph.png"))

## 4. Download model images

In [ ]:
files.download("discovered_process_model.png")
files.download("directly_follows_graph.png")

## Troubleshooting
If installation or a PM4Py API call fails, rerun the install cell and restart the runtime if needed. Do not fabricate timestamps. If the diagram is too dense, investigate a documented subset or filter infrequent activities and clearly state the filter.